# Pruned Perfect Play — Paw Graph $P_w$

Same min–max + ES-pruning engine as `perfect_play_c3.ipynb`, but applied to the **paw graph** as target:

$$ P_w = K_3 \text{ with a pendant edge:\ } \{v_1, v_2, v_3\} \text{ triangle},\ v_4 \text{ joined to } v_2. $$

Density: $m(P_w) = \max\{3/2,\, 2\} = 2$, attained on the triangle subgraph, so by Bednarska–Łuczak
$$ q^{*}(n, P_w) = \Theta(\sqrt{n}). $$
Same asymptotic order as $q^{*}(n, K_3)$, but with a smaller constant since building a paw is strictly harder than building a triangle.

Number of paw subgraphs in $K_n$: $12 \binom{n}{4}$ (4 triangles per 4-subset × 3 pendant attachments).

In [ ]:
import math
import time
from itertools import combinations, permutations
import matplotlib.pyplot as plt
import numpy as np

IITR_BLUE   = "#1f497d"
IITR_ORANGE = "#df802a"
plt.rcParams["figure.dpi"] = 110

def edge(u, v):
    return frozenset({u, v})

## Generating all paws in $K_n$

For each 4-subset $\{a, b, c, d\}$ of vertices, the 12 paws are: pick the triangle (4 ways), pick the pendant attachment (3 ways).

In [ ]:
def paws(n):
    """All paw subgraphs in K_n as frozensets of 4 edges."""
    out = set()
    for v1, v2, v3, v4 in combinations(range(n), 4):
        # Choose which 3 of the 4 vertices form the triangle
        for tri in combinations((v1, v2, v3, v4), 3):
            pendant_vertex = (set([v1, v2, v3, v4]) - set(tri)).pop()
            tri_edges = {edge(a, b) for a, b in combinations(tri, 2)}
            # Attach pendant to each triangle vertex (3 options)
            for attach in tri:
                paw = frozenset(tri_edges | {edge(attach, pendant_vertex)})
                out.add(paw)
    return list(out)

# Sanity check: |paws(n)| should be 12 * C(n, 4)
for n in [4, 5, 6, 7]:
    expected = 12 * math.comb(n, 4)
    actual = len(paws(n))
    print(f"n={n}: expected {expected}, got {actual}, {'OK' if expected == actual else 'MISMATCH'}")

## Search engine (identical to `perfect_play_c3.ipynb`)

Min–max with memoisation, ES pruning at start of Breaker's turn, optional move ordering by $|\Delta \Phi_{ES}|$.

In [ ]:
class Game:
    def __init__(self, F, n, q, pruning=True, order=True):
        self.F = tuple(frozenset(A) for A in F)
        self.n = n
        self.q = q
        self.pruning = pruning
        self.order = order
        self.all_edges = frozenset(edge(u, v) for u, v in combinations(range(n), 2))
        self.cache = {}
        self.states_explored = 0

    def maker_won(self, M):
        return any(A <= M for A in self.F)

    def breaker_won(self, B):
        return all(A & B for A in self.F)

    def phi_es(self, M, B):
        s = 0.0
        for A in self.F:
            if A & B:
                continue
            s += 2 ** (len(A & M) - len(A))
        return s

    def order_moves(self, free, M, B, who):
        if not self.order:
            return list(free)
        base = self.phi_es(M, B)
        def delta(e):
            return (self.phi_es(M | {e}, B) - base) if who == "M" \
                   else (base - self.phi_es(M, B | {e}))
        return sorted(free, key=lambda e: -abs(delta(e)))

    def value(self, M, B, turn, b_left):
        key = (M, B, turn, b_left)
        if key in self.cache:
            return self.cache[key]
        self.states_explored += 1

        if self.maker_won(M):
            self.cache[key] = "M"; return "M"
        if self.breaker_won(B):
            self.cache[key] = "B"; return "B"

        free = self.all_edges - M - B
        if not free:
            self.cache[key] = "B"; return "B"

        if turn == "M":
            result = "B"
            for e in self.order_moves(free, M, B, "M"):
                new_M = M | {e}
                if self.maker_won(new_M):
                    result = "M"; break
                next_turn, next_left = ("M", 0) if self.q == 0 else ("B", self.q)
                if self.value(new_M, B, next_turn, next_left) == "M":
                    result = "M"; break
        else:
            if self.pruning and b_left == self.q:
                if self.phi_es(M, B) < 1:
                    self.cache[key] = "B"; return "B"
            result = "M"
            for e in self.order_moves(free, M, B, "B"):
                new_B = B | {e}
                if self.breaker_won(new_B):
                    sub = "B"
                elif b_left == 1:
                    sub = self.value(M, new_B, "M", self.q)
                else:
                    sub = self.value(M, new_B, "B", b_left - 1)
                if sub == "B":
                    result = "B"; break
        self.cache[key] = result
        return result

    def play(self):
        return self.value(frozenset(), frozenset(), "M", self.q)


def threshold(F, n, pruning=True, order=True, q_max=None, time_limit=300):
    if q_max is None:
        q_max = math.comb(n, 2)
    states = {}
    timings = {}
    for q in range(1, q_max + 1):
        t0 = time.time()
        g = Game(F, n, q, pruning=pruning, order=order)
        winner = g.play()
        states[q] = g.states_explored
        timings[q] = time.time() - t0
        if timings[q] > time_limit:
            return None, states, timings
        if winner == "B":
            return q, states, timings
    return q_max + 1, states, timings

## Exact thresholds $q^{*}(n, P_w)$ for $n = 4 \ldots 7$

In [ ]:
ns = [4, 5, 6, 7]    # set to [4, 5, 6, 7, 8] for a longer run
paw_thresholds = {}
paw_states_pruned = {}
paw_states_plain  = {}

for n in ns:
    F = paws(n)
    print(f"n={n}: |F| = {len(F)} paws,  computing pruned ...", end=" ", flush=True)
    t0 = time.time()
    q_star, sp, _ = threshold(F, n, pruning=True, order=True)
    print(f"q* = {q_star} (in {time.time()-t0:.1f}s)")
    paw_thresholds[n] = q_star
    paw_states_pruned[n] = sp
    if n <= 5:
        _, spl, _ = threshold(F, n, pruning=False, order=False, q_max=q_star)
        paw_states_plain[n] = spl
    else:
        paw_states_plain[n] = None

## Compare with the triangle game

We re-run the triangle thresholds in this notebook for a clean side-by-side comparison.

In [ ]:
def triangles(n):
    return [frozenset({edge(a, b), edge(b, c), edge(a, c)})
            for a, b, c in combinations(range(n), 3)]

tri_thresholds = {}
for n in ns:
    F = triangles(n)
    q_star, _, _ = threshold(F, n, pruning=True, order=True)
    tri_thresholds[n] = q_star
    print(f"n={n}: q*(K_3) = {q_star}")

In [ ]:
# Side-by-side table
print(f"{'n':>3} {'sqrt(n)':>8} {'q*(K_3)':>9} {'q*(P_w)':>9} {'gap':>5}")
print("-" * 40)
for n in ns:
    gap = tri_thresholds[n] - paw_thresholds[n]
    print(f"{n:>3} {math.sqrt(n):>8.2f} {tri_thresholds[n]:>9} "
          f"{paw_thresholds[n]:>9} {gap:>5}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.array(ns)
ax.plot(x, [tri_thresholds[n] for n in ns], "o-", color=IITR_BLUE,
        linewidth=2.5, markersize=10, label=r"$q^{*}(n, K_3)$")
ax.plot(x, [paw_thresholds[n] for n in ns], "s-", color=IITR_ORANGE,
        linewidth=2.5, markersize=10, label=r"$q^{*}(n, P_w)$")
xs_smooth = np.linspace(min(ns)-0.5, max(ns)+0.5, 200)
ax.plot(xs_smooth, np.sqrt(xs_smooth), "--", color="grey", alpha=0.7,
        label=r"$\sqrt{n}$ reference")
ax.set_xlabel("n"); ax.set_ylabel("q")
ax.set_title("Exact thresholds: Triangle vs Paw", color=IITR_BLUE)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## States explored — pruned vs plain

In [ ]:
rows = []
for n in ns:
    q_play = max(1, paw_thresholds[n] - 1)
    pruned = paw_states_pruned[n].get(q_play, "--")
    plain = paw_states_plain[n].get(q_play, "--") if paw_states_plain[n] is not None else "--"
    speedup = (f"{plain / pruned:.0f}×"
               if isinstance(plain, int) and isinstance(pruned, int) and pruned > 0
               else "--")
    rows.append((n, paw_thresholds[n], q_play, plain, pruned, speedup))

print(f"{'n':>3} {'q*':>4} {'q played':>9} {'plain':>14} {'pruned':>14} {'speedup':>10}")
print("-" * 60)
for r in rows:
    n, qs, qp, pl, pr, sp = r
    pl_s = f"{pl:.2e}" if isinstance(pl, int) else str(pl)
    pr_s = f"{pr:.2e}" if isinstance(pr, int) else str(pr)
    print(f"{n:>3} {qs:>4} {qp:>9} {pl_s:>14} {pr_s:>14} {sp:>10}")

## Observations

- $q^{*}(n, P_w) \leq q^{*}(n, K_3)$ for every tested $n$: building a paw is strictly harder than building a triangle, since the pendant edge $v_2 v_4$ adds an extra constraint on top of triangle construction.
- Both grow as $\Theta(\sqrt{n})$, confirming the Bednarska–Łuczak prediction (the densest subgraph of $P_w$ is its triangle, so $m(P_w) = m(K_3) = 2$).
- The gap is at most 1 in this range — consistent with the constant factor being slightly smaller for the paw.

**Computational note.** The paw winning-set family is much larger than the triangle family ($12\binom{n}{4}$ vs $\binom{n}{3}$ — about $\frac{n-3}{2} \cdot 12$ times more sets at the same $n$). Each ES-potential evaluation does work proportional to $|\mathcal F|$, so the paw search is roughly an order of magnitude slower per state than the triangle search at the same $n$.